#  Inference & Decoding

## What happens during inference?

Inference is the process where a pretrained or fine-tuned LLM generates outputs in response to user input (prompt). It’s the stage where the model is applied in real-time rather than being trained.

During inference, the input text is first tokenized into words, subwords, or bytes. These tokens are converted into embeddings and, if using a Transformer, positional encodings are added to retain word order. The embeddings are passed through the model layers, where self-attention and feed-forward networks compute contextual representations.

The model predicts the next token at each step, often using autoregressive decoding. Depending on the strategy (greedy, beam search, sampling, or top-k/top-p), the model selects the most likely token or a probabilistic choice. This process repeats until an end-of-sequence token is generated or a maximum length is reached.

Optionally, safety layers, guardrails, or RAG retrieval steps may be applied to:

- Ensure outputs are safe and aligned.

- Retrieve external knowledge to improve factual accuracy.

- Filter out toxic or irrelevant responses.

Finally, the generated tokens are detokenized back into human-readable text, forming the model’s output.

## Greedy Decoding


Greedy decoding is a simple and deterministic strategy for generating text from LLMs where, at each step, the model selects the most probable next token without considering alternatives.

How It Works

During inference, the model predicts a probability distribution over all possible next tokens. Greedy decoding always picks the token with the highest probability, then feeds it back into the model to predict the next token. This process continues until the end-of-sequence token is generated or a maximum length is reached.

Pros

- ✅ Simple and fast to compute.

- ✅ Deterministic: same input always gives the same output.

- ✅ Works well for tasks needing precise, factual responses.

Cons

- ⚠️ Can produce repetitive or generic outputs.

- ⚠️ Lacks diversity and may miss more creative or contextually better alternatives.

- ⚠️ Can get trapped in loops for long sequences.

Analogy

Greedy decoding is like always taking the most obvious path at every crossroads:

- You reach a destination quickly.

- But you may miss more interesting or better routes.

## Beam Search


Beam search is a decoding strategy used in LLMs that balances accuracy and diversity by keeping track of multiple candidate sequences (beams) at each step instead of only the most probable token.

How It Works

During inference, the model predicts a probability distribution for the next token. Instead of selecting just the highest-probability token (like greedy decoding), beam search maintains the top k sequences (beams) at each step. For each sequence, all possible next tokens are considered, and the top k overall sequences are kept for the next step. This continues until all sequences reach the end-of-sequence token or a maximum length. Finally, the sequence with the highest overall probability is selected as output.

Pros

- ✅ Produces higher-quality and more coherent outputs than greedy decoding.

- ✅ Reduces the chance of early mistakes leading to poor sequences.

- ✅ Allows control over diversity vs. precision by adjusting beam width.

Cons

- ⚠️ More computationally expensive than greedy decoding.

- ⚠️ Large beam widths can still produce generic or repetitive text.

- ⚠️ Does not fully guarantee creative or optimal outputs.

Analogy

Beam search is like exploring the top few paths at every crossroads:

- Instead of always taking the most obvious route, you keep several promising options open.

- This increases the chance of finding a better overall path while still being efficient.

## Top-k Sampling


Top-k sampling is a probabilistic decoding strategy where, at each step, the model considers only the k most probable tokens and samples the next token from this limited set, instead of always picking the highest-probability token.

How It Works

During inference, the model predicts a probability distribution over the entire vocabulary. The top k tokens with highest probabilities are selected, and all other tokens are discarded. The next token is randomly sampled from this top-k set according to their normalized probabilities. This allows the model to generate more diverse and creative outputs while avoiding extremely unlikely tokens.

Pros

- ✅ Introduces controlled randomness, making outputs more diverse and creative.

- ✅ Avoids the repetitive or overly deterministic behavior of greedy decoding.

- ✅ Provides a simple way to trade-off quality vs. creativity by adjusting k.

Cons

- ⚠️ The choice of k affects performance:

    - Too small → outputs become deterministic and repetitive.

    - Too large → outputs may become nonsensical or incoherent.

- ⚠️ Still probabilistic → outputs are non-deterministic for the same prompt.

Analogy

Top-k sampling is like choosing your next move from the top few options in a game rather than always picking the “safest” move:

- You introduce variety and creativity while avoiding extremely poor options.

## Top-p (Nucleus) Sampling

Top-p sampling, also called nucleus sampling, is a probabilistic decoding strategy where, at each step, the model considers the smallest set of tokens whose cumulative probability exceeds p and samples the next token from this set.

How It Works

During inference, the model predicts a probability distribution over the vocabulary. Instead of picking the top k tokens like in Top-k sampling, top-p selects a dynamic set of tokens whose cumulative probability is ≥ p (e.g., 0.9). The next token is then randomly sampled from this “nucleus” set. This allows the model to adaptively choose a variable number of candidate tokens, balancing diversity and coherence.

Pros

- ✅ Introduces controlled randomness for more natural and creative outputs.

- ✅ Avoids the need to fix k as in Top-k, adapting to different probability distributions.

- ✅ Reduces the chance of sampling low-probability, nonsensical tokens.

Cons

- ⚠️ Still probabilistic → outputs vary between runs.

- ⚠️ Choosing p requires tuning for each task: too high → incoherent outputs, too low → deterministic.

- ⚠️ Slightly more computationally intensive than greedy decoding.

Analogy

Top-p sampling is like picking from the most promising group of options in a game rather than a fixed number of top moves:

- The size of the group changes depending on the situation.

- You maintain creativity while avoiding very unlikely or bad choices.

## Temperature in LLMs


Temperature is a hyperparameter in probabilistic decoding that controls the randomness of token selection. It scales the logits (predicted token probabilities) before applying softmax, influencing how creative or deterministic the model’s output is.

How It Works

During inference, the model produces a probability distribution over the vocabulary. The logits are divided by the temperature T before softmax:

- Low Temperature (T < 1): Sharpens the distribution → higher-probability tokens are favored → outputs are more deterministic and focused.

- High Temperature (T > 1): Flattens the distribution → lower-probability tokens are more likely → outputs are more diverse and creative.

- T = 1: No scaling; standard probabilities are used.

Pros

- ✅ Controls creativity vs. determinism in outputs.

- ✅ Works well with sampling methods (Top-k, Top-p) to balance diversity.

- ✅ Simple and widely used hyperparameter tweak.

Cons

- ⚠️ Too low → outputs become repetitive or generic.

- ⚠️ Too high → outputs may be nonsensical or incoherent.

- ⚠️ Requires tuning per task or model to get optimal results.

Analogy

Temperature is like adjusting the “risk level” for a chef improvising a recipe:

- Low temperature → chef sticks to safe, predictable ingredients.

- High temperature → chef experiments with unusual combinations, creating more diverse dishes.

## Repetition Penalty

Repetition penalty is a mechanism during decoding that discourages the model from repeating the same token or phrase multiple times. It is commonly used with sampling methods to make generated text more coherent and natural.

How It Works

During inference, the model predicts the probability of the next token. Tokens that have already been generated are penalized by reducing their probabilities. This discourages the model from selecting the same tokens repeatedly. The strength of the penalty is controlled by a hyperparameter, which adjusts how strongly repeated tokens are suppressed.

Pros

- ✅ Reduces repetitive and looping outputs.

- ✅ Improves fluency and readability of generated text.

- ✅ Works well with probabilistic decoding methods like Top-k, Top-p, and temperature sampling.

Cons

- ⚠️ Too high a penalty → may over-penalize valid repetitions in natural language (e.g., “very, very”).

- ⚠️ Adds another hyperparameter to tune for optimal output.

- ⚠️ Does not completely eliminate repetition if the model strongly favors certain patterns.

Analogy

Repetition penalty is like a coach telling a musician not to repeat the same note too often:

- The musician still plays the melody but avoids excessive repetition.

- This makes the performance more pleasant and engaging.

## Streaming Responses

Streaming responses is a technique in LLM inference where partial outputs are sent to the user in real-time, as the model generates them token by token, rather than waiting for the entire sequence to be completed.

How It Works

During inference, the model predicts the next token sequentially. Instead of buffering the full output until the end, each generated token or small chunk is immediately sent to the client. This allows applications, such as chatbots or virtual assistants, to display answers progressively, giving a sense of faster response and interactivity.

Streaming can be combined with sampling strategies, guardrails, and repetition penalties, ensuring that outputs remain coherent, safe, and aligned, even when sent incrementally.

Importance

- Improves user experience with faster visible feedback.

- Enables interactive applications where partial outputs inform the next user action.

- Useful for long-form generation where waiting for full completion would be slow.

Pros

- ✅ Reduces perceived latency for users.

- ✅ Supports interactive and conversational applications.

- ✅ Works with both deterministic and probabilistic decoding.

Cons

- ⚠️ Requires careful token handling to maintain output coherence.

- ⚠️ Early partial outputs may mislead users if not fully generated.

- ⚠️ Slightly more complex to implement compared to full-sequence generation.

Analogy

- Streaming responses are like watching a painter create a mural live:

- You see the artwork gradually as it’s being created.

- You don’t wait until the end to appreciate or interact with it.

## Deterministic vs Stochastic Outputs

Deterministic outputs always produce the same response for a given input. Models use strategies like greedy decoding or beam search to select the most probable token at each step, resulting in predictable and repeatable outputs. Stochastic outputs introduce controlled randomness, using methods like Top-k, Top-p (nucleus) sampling, or temperature scaling, so the same input can generate different outputs each time.

Deterministic outputs are reliable and consistent, making them ideal for factual tasks like summarization, code generation, or question answering. Stochastic outputs are creative and diverse, suitable for storytelling, dialogue, or brainstorming, where variety is desired.

Deterministic outputs are simple to implement and fast, but can be generic, repetitive, or overly safe. Stochastic outputs require additional hyperparameter tuning (temperature, top-k, or top-p) and may produce inconsistent or less factual results.

Deterministic methods are used in tasks where accuracy and repeatability are critical. Stochastic methods are preferred when creativity, exploration, or diversity is needed in model outputs.

Deterministic outputs give predictable, safe responses. Stochastic outputs give varied and creative responses, allowing models to explore multiple plausible continuations.